In [ ]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from datetime import date

import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter

import eurostat #python wrapper for taking data. 
import time

In [ ]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data\Storage'

In [ ]:
import os
os.chdir(Share_point)

In [ ]:
os.getcwd()

In [ ]:
Points_APIs = pd.read_excel(r'Storage_facilities_API_call.xlsx')

### Version 5 of the API calls GIE

In [ ]:
# AGI_APIs = pd.read_excel(r'C:\Users\giovanni.sgaravatti\OneDrive - Bruegel\Documents\Gas flow model\AGSI storage\SSO_EIC excel.xlsx')

In [ ]:
# Points_APIs = pd.read_excel(r'C:\Users\giovanni.sgaravatti\OneDrive - Bruegel\Documents\Gas flow model\AGSI storage\SSO_EIC 30.11.2021 excel.xlsx',sheet_name='Sheet2')
# Points_APIs.head(2)
# Points_APIs.iloc[1,6]
# headers = {"x-key":"c004438249338e49ed35e0ffce94cb6a"}

####### Date 
2022-02-25 00:00:00

We are pleased to announce that EDF Gas Deutschland GmbH and EnBW Etzel Speicher GmbH are joining AGSI+ for publication of their share in the jointly operated storage facility in Etzel, Germany.

This joint publication replaces the dataset entry for Crystal, Friedeburger Speicherbetriebsgesellschaft mbH (EIC 21X000000001125L) previously listed for UGS Oude Statenzijl Etzel Crystal (EIC 21W000000000013E)

Crystal is the Joint Venture between EDF Gas Deutschland GmbH and EnBW Etzel Speicher GmbH to operate the storage facility in Etzel, Germany.

The designated SSOs (EDF Gas Deutschland GmbH and EnBW Etzel Speicher GmbH) are each reporting for their share in the Etzel storage facility.

### Version 6 of the API calls GIE

In [ ]:
headers = {"x-key":"d8561648296cb38e6c400823755689941530"} # After July 4th 2022

### Can skip now

In [ ]:
url_list = 'http://agsi.gie.eu/api/about?show=listing'
r_list = requests.get(url_list,headers=headers)
storage_list = r_list.json()

In [ ]:
name = []
UGS_type = []
eic =[]
country = []
company = []
url = []

In [ ]:
for item in storage_list:
    for facility in item['facilities']:
        name_f = facility['name']
        UGS_type_f = facility['type']
        eic_f = facility['eic']
        country_f = facility['country']
        company_f = facility['company']
        url_f = facility['url']
        
        name.append(name_f)
        UGS_type.append(UGS_type_f)
        eic.append(eic_f)
        country.append(country_f)
        company.append(company_f)
        url.append(url_f)

In [ ]:
storage_list_df = pd.DataFrame()
storage_list_df['name']=name
storage_list_df['type']=UGS_type
storage_list_df['eic']=eic
storage_list_df['country']=country
storage_list_df['company_eic']=company
storage_list_df['url']=url

In [ ]:
storage_list_df.to_excel('Storage_facilities_API_call.xlsx')

### Restart here

In [ ]:
dates = []
values = []
injection = []
withdrawal =[]
percentage = []
status = []
max_storage = []
country = []
Name = []

for item in range(len(Points_APIs)):
        url = Points_APIs.iloc[item,6]+'&from=2022-07-01&size=300'
        try:
            r = requests.get(url,headers=headers)
            if r.status_code != 200:
                print(r.status_code)
                print(url)

            raw_data = r.json()
            inner_data = raw_data['data']
            for x in inner_data:
                #here we work with the APIs
#                 date = x['gasDayStartedOn']     # Changed on 5 April 2022
                date = x['gasDayStart']
                value = x['gasInStorage']
                inj = x['injection']
                withdr = x['withdrawal']
                percentages = x['full']
                statuses = x['status']
                Max_Storage = x['workingGasVolume']
                #here we work with the excel file
                countries = Points_APIs.iloc[item,4]
                Names = Points_APIs.iloc[item,1]


                dates.append(date)     
                values.append(value)
                injection.append(inj)
                withdrawal.append(withdr)
                percentage.append(percentages)
                status.append(statuses)
                max_storage.append(Max_Storage)
                country.append(countries)
                Name.append(Names)
        except Exception as e:
            print(e)

In [ ]:
df = pd.DataFrame()
df['dates'] = dates
df['values'] = values
df['injection'] = injection
df['withdrawal'] = withdrawal
df['fillness'] = percentage
df['status'] = status
df['Max Storage'] = max_storage
df['country'] = country
df['name'] = Name

In [ ]:
# Getting rid of NaNa
df.replace('', np.NaN)
# Getting rid of observations for which we have no data
df = df[df['status']!='N']

In [ ]:
df['values'] = pd.to_numeric(df['values']) 
df['fillness'] = pd.to_numeric(df['fillness'])
df['Max Storage'] = pd.to_numeric(df['Max Storage']) 

In [ ]:
from datetime import date
df.to_csv('Storage all from 2015/historic data/2022_2.csv')

###  merge with historic data

In [ ]:
# df = pd.read_csv(r'C:\Users\giovanni.sgaravatti\Documents\Gas flow model\AGSI storage\storage all.csv')
# df = pd.read_csv('storage all from 2015/{}.csv'.format(date.today()))

In [ ]:
# Change current directory
import os
os.chdir(Share_point + '\Storage all from 2015\historic data')

In [ ]:
df_1a = pd.read_csv('2015_1.csv',index_col=0)
df_1b = pd.read_csv('2015_2.csv',index_col=0)
df_2a = pd.read_csv('2016_1.csv',index_col=0)
df_2b = pd.read_csv('2016_2.csv',index_col=0)
df_3a = pd.read_csv('2017_1.csv',index_col=0)
df_3b = pd.read_csv('2017_2.csv',index_col=0)
df_4a = pd.read_csv('2018_1.csv',index_col=0)
df_4b = pd.read_csv('2018_2.csv',index_col=0)
df_5a = pd.read_csv('2019_1.csv',index_col=0)
df_5b = pd.read_csv('2019_2.csv',index_col=0)
df_6a = pd.read_csv('2020_1.csv',index_col=0)
df_6b = pd.read_csv('2020_2.csv',index_col=0)
df_7a = pd.read_csv('2021_1.csv',index_col=0)
df_7b = pd.read_csv('2021_2.csv',index_col=0)
df_8a = pd.read_csv('2022_1.csv',index_col=0)
df_8b = pd.read_csv('2022_2.csv',index_col=0)

In [ ]:
df = pd.concat([df_1a,df_1b,df_2a,df_2b,df_3a,df_3b,df_4a,df_4b,df_5a,df_5b,df_6a,df_6b,df_7a,df_7b,df_8a,df_8b])

In [ ]:
# del df['Unnamed: 0']
df = df.drop_duplicates()

In [ ]:
# This doesn't work Unable to parse string "-" at position 11862
# Getting rid of observations for which we have no data
#df = df[(df['status']!='N') & (df['status']!='E')]
df = df[(df['status']!='N')]
# Getting rid of NaNa
df.replace('', np.NaN)  #we try to set missing values to the numpy notation, NaN - easier to work with. 
df.replace('-', np.NaN)
# #We need to set it to integer to work with. 
#Using coerce we actually set missing values to the numpy notation, NaN - easier to work with. 
df['values'] = pd.to_numeric(df['values'],errors='coerce')  
df['injection'] = pd.to_numeric(df['injection'],errors='coerce') 
df['withdrawal'] = pd.to_numeric(df['withdrawal'],errors='coerce') 
df['fillness'] = pd.to_numeric(df['fillness'],errors='coerce') 
df['Max Storage'] = pd.to_numeric(df['Max Storage'],errors='coerce') 

In [ ]:
df[(df['name'] == 'UGS Katharina')& (df['dates']=='2023-01-22')]

In [ ]:
non_EU = ['UA','RS']

In [ ]:
# Getting rid of non-EU countries, Ukraine shows crazy numbers
df=df[~df.country.isin(non_EU)]

In [ ]:
df = df.set_index(pd.DatetimeIndex(df['dates']))

In [ ]:
del df['dates']

In [ ]:
df = df.sort_values(by='dates')

In [ ]:
df

# Gazprom

In [ ]:
# Change current directory
import os
os.chdir(Share_point)

Gazprom holds stocks at 
Haidach in Austria (via its wholly owned subsidiaries, Astora and GSA),
Dambořice in the Czech Republic (via its subsidiary, Moravia Gas Storage), 
Bergermeer in the Netherlands (where Gazprom holds 19 TWh of capacity in the 48 TWh capacity facility through partnership with TAQA), 
and at Jemgum and Rehden (via Astora) in Germany. 

In addition, Gazprom has access to half the capacity at
Katharina in Germany (via Astora) and
one third of the capacity held at Etzel in Germany by EKB (Gazprom being a 33 per cent shareholder in EKB).

Data on AGSI+ are provided in Twh therefore to convert in M3m we have to multiply by103

In [ ]:
Gazprom_grouped = ['UGS Jemgum H (astora)', 'UGS Jemgum H (VGS)',
                   'UGS Jemgum H (EWE)', # 26.04 addition 
                   #'UGS Katharina',
                   'UGS Rehden',
                   'UGS Haidach (astora)', 'UGS Haidach (GSA)',
                   'UGS Dambořice']

In [ ]:
EU_Gazprom = df[df['name'].isin(Gazprom_grouped)]
EU_Gazprom['name'].unique()

In [ ]:
EU_Gazprom.tail(7)

In [ ]:
EU_Gazprom.to_excel('Gazprom daily/EU_Gazprom_raw.xlsx')

In [ ]:
#This creates dataframe for looking at Gazprom in TWh. 
EU_Gazprom = EU_Gazprom.groupby(["dates"])["values",'injection','withdrawal','Max Storage'].sum().reset_index()
EU_Gazprom = EU_Gazprom.set_index(pd.DatetimeIndex(EU_Gazprom['dates']))

In [ ]:
del EU_Gazprom['dates']

In [ ]:
EU_Gazprom.groupby(["dates"])["values",'injection','withdrawal','Max Storage'].sum().reset_index()

In [ ]:
#check for inconsistencies in the graph
EU_Gazprom['daily flow'] = EU_Gazprom['injection']-EU_Gazprom['withdrawal'] 

In [ ]:
# EU_Gazprom['daily flow']['2021-11-22':'2021-12-05']
# last_data = EU_Gazprom[EU_Gazprom['dates']>='2021-11-15']

In [ ]:
converter=0.0103
converter_GWh = converter/1000

In [ ]:
EU_Gazprom['storage in M3m'] = EU_Gazprom['values']/converter
EU_Gazprom['injection in M3m'] = EU_Gazprom['injection']/converter_GWh
EU_Gazprom['withdrawal in M3m'] = EU_Gazprom['withdrawal']/converter_GWh
EU_Gazprom['daily flow in M3m'] = EU_Gazprom['daily flow']/converter_GWh
EU_Gazprom['Max Storage in M3m'] = EU_Gazprom['Max Storage']/converter

In [ ]:
EU_Gazprom['Max Storage in M3m']['2022-05-19']

In [ ]:
# Daily data for Gazprom

In [ ]:
from datetime import date
EU_Gazprom.to_csv('Gazprom daily/Daily_flows_Gazprom {}.csv'.format(date.today()))
# last_data.to_csv('Gazprom last data by point.csv')

In [ ]:
# Weekly data

In [ ]:
GazpromPlot = EU_Gazprom.groupby(pd.Grouper(freq='W')).mean()

In [ ]:
# GazpromPlot.to_csv('Gazprom weekly\weekly_flows_Gazprom {}.csv'.format(date.today()))

In [ ]:
import os
os.getcwd()

## Graphs

# Last 30 days comparison

In [ ]:
del EU_Gazprom['injection']
del EU_Gazprom['withdrawal']
del EU_Gazprom['daily flow']
del EU_Gazprom['storage in M3m'] 
del EU_Gazprom['injection in M3m']
del EU_Gazprom['withdrawal in M3m']
del EU_Gazprom['daily flow in M3m']
del EU_Gazprom['Max Storage in M3m']
del EU_Gazprom['Max Storage']

### Old values (2015-2020)

In [ ]:
# Here we get the dates and values for the last 30 days in all last 5 years (ex. 5 Oct to 5 Nov in 2015 to 2020)

gas = []
dates = []

from datetime import date
today = date.today()
month_str = str(today.month)
day_str = str(today.day)
from pandas.tseries.offsets import DateOffset

# for year in ['2015','2016','2017','2018','2019','2020']: ### not working cause we don't have data for 2014
for year in ['2016','2017','2018','2019','2020']:
    old_date = year+'-'+month_str+'-'+day_str
    old_date = pd.to_datetime(old_date)
    month_int = int(today.month)
#     date_less_30 = old_date - DateOffset(month=month_int-1)   ### not working in January
    date_less_30 = old_date - timedelta(days=30)
    old_date = old_date.date()
    date_less_30 = date_less_30.date()
    old_values_Mm3 = EU_Gazprom[date_less_30:old_date]['values']
    date_index = EU_Gazprom[date_less_30:old_date].index #['dates']
    gas.append(old_values_Mm3)
    dates.append(date_index)

In [ ]:
# n_years = [0,1,2,3,4,5]
n_years = [0,1,2,3,4]
n_days = [*range(0, 30, 1)]

values = []
days = []

for i in n_years:
    for j in n_days:
        values.append(gas[i][j])
        days.append(dates[i][j])

In [ ]:
d = {'days':days,'values':values}
df_old_Gazprom = pd.DataFrame(d)

In [ ]:
# 5-year average dataframe
df_old_Gazprom = df_old_Gazprom.set_index(pd.DatetimeIndex(df_old_Gazprom['days']))
del df_old_Gazprom['days']
df_old_Gazprom['day and month'] = df_old_Gazprom.index.astype(str).str[5:]

In [ ]:
# minvalues_Mm3 = df_old.groupby("daily").min()/converter
# maxvalues_Mm3 = df_old.groupby("daily").max()/converter
avgvalues = df_old_Gazprom.groupby("day and month").mean()/converter #last 5-year average for Gazprom daily data

### Last 30 days

In [ ]:
# I create a new dataframe to work with it later and to distinguish it from the one with old values (5-year avgs)
EU_Gazprom_30 = EU_Gazprom.iloc[-31:]
# I add a daily column, like for the otehr dataframe
EU_Gazprom_30['day and month'] = EU_Gazprom_30.index.astype(str).str[5:]
#convert from TWh to M3m
EU_Gazprom_30['M3m values'] = EU_Gazprom_30['values']/converter

In [ ]:
# del EU_Gazprom_30['dates']
del EU_Gazprom_30['values']

In [ ]:
values_30_days = EU_Gazprom_30.groupby("day and month").max()

In [ ]:
per_of_avg = (values_30_days['M3m values']/avgvalues['values'])*100
per_of_avg = per_of_avg.to_frame()
# Rename the column but unsuccessfully
per_of_avg.rename(columns={per_of_avg.columns[0]: 'Ratio'}, errors="raise", inplace=True)

In [ ]:
# Create a Pandas Excel writer using XlsxWriter as the engine and save average and 2021 values.
from datetime import date

writer = pd.ExcelWriter(r'Gazprom daily/Gazprom {}.xlsx'.format(date.today()), engine='xlsxwriter')

# Write each dataframe to a different worksheet.
avgvalues.to_excel(writer, sheet_name='average')
values_30_days.to_excel(writer, sheet_name='2021')
per_of_avg.to_excel(writer, sheet_name='gazprom_2021_ratio')
EU_Gazprom_30.iloc[-30:].to_excel(writer, sheet_name='dataset')
writer.save()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# ax.plot(minvalues_Mm3,label='Minimum')
# ax.plot(maxvalues_Mm3,label='Maximum')
ax.plot(avgvalues,label='Average')
ax.plot(values_30_days['M3m values'],label='2022')

plt.title('Gazprom Gas Storage now vs average of same 30 days 2015-2020',fontsize='16')
plt.ylabel('Mm3',fontsize=12,rotation=0,labelpad=25)
plt.xlabel('Last 30 days',fontsize=12,rotation=0)
tick_val = [1,5,10,15,20,25,30]
plt.xticks(tick_val)
plt.legend()
plt.show()
fig.savefig('figures/Gazprom storage.png',dpi=300,bbox_inches="tight")

# All series by week/month

In [ ]:
GazpromPlot = EU_Gazprom.groupby(pd.Grouper(freq='W')).mean()

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(18.5, 10.5)
for yr in ['2017','2018','2019','2020','2021','2022','2023']:
    ax.plot(GazpromPlot.loc[yr].values, label = 'Year {}'.format(yr))
    plt.legend(loc='best')

# Gazprom charts min/max 2021 both in M3m and TWh

In [ ]:
#compare by week or month
compare = 'weekly'

In [ ]:
GazpromPlot[compare] = GazpromPlot.index.isocalendar().week

In [ ]:
from datetime import date
today = date.today()
minvalues_Mm3 = GazpromPlot['2015-01-01':'2019-12-29'].groupby(compare).min()/converter
maxvalues_Mm3 = GazpromPlot['2015-01-01':'2019-12-29'].groupby(compare).max()/converter
avgvalues = GazpromPlot['2015-01-01':'2019-12-29'].groupby(compare).mean()/converter
# values2021_Mm3 = GazpromPlot['2020-12-28':'{}'.format(today)].groupby(compare).last()/converter
values2021_Mm3 = GazpromPlot['2020-12-28':'2022-01-02'].groupby(compare).last()/converter  #from week 1 to 53 of 2021
values2022_Mm3 = GazpromPlot['2022-01-03':'2023-01-01'].groupby(compare).last()/converter  #from week 1 to 53 of 2021
values2023_Mm3 = GazpromPlot['2023-01-02':].groupby(compare).last()/converter  #from week 1 to 53 of 2021

In [ ]:
# values2022_Mm3
values2023_Mm3

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# ax.plot(minvalues_Mm3,label='Minimum')
# ax.plot(maxvalues_Mm3,label='Maximum')
ax.plot(avgvalues,label='Average')
ax.plot(values2021_Mm3,label='2021')
ax.plot(values2022_Mm3,label='2022')
ax.plot(values2023_Mm3,label='2023')

plt.title('Gazprom Gas Storage 2021 compared with 2015-2020 average',fontsize='16')
plt.ylabel('Mm3',fontsize=12,rotation=0,labelpad=25)
plt.xlabel('Week Number',fontsize=12,rotation=0)
plt.legend()
plt.show()
fig.savefig('figures/Gazprom storage.png',dpi=300,bbox_inches="tight")

In [ ]:
# Create a Pandas Excel writer using XlsxWriter as the engine and save average and 2021 values.
# from datetime import date
# writer = pd.ExcelWriter(r'Gazprom weekly/Gazprom Storage weekly {}.xlsx'.format(date.today()), engine='xlsxwriter')

# # Write each dataframe to a different worksheet.
# minvalues_Mm3.round(1).to_excel(writer, sheet_name='Min')
# maxvalues_Mm3.round(1).to_excel(writer, sheet_name='Max')
# values2021_Mm3.round(1).to_excel(writer, sheet_name='2021')
# values2022_Mm3.round(1).to_excel(writer, sheet_name='2022')

# writer.save()
# writer.close()

#### Save excel for infogram

In [ ]:
GZ_weekly = pd.DataFrame()
GZ_weekly['minvalues'] = minvalues_Mm3.round(1)
GZ_weekly['maxvalues'] = maxvalues_Mm3.round(1)
GZ_weekly['2021'] = values2021_Mm3.round(1)
GZ_weekly['2022'] = values2022_Mm3.round(1)
GZ_weekly['2023'] = values2023_Mm3.round(1)
GZ_weekly.to_excel(r'Gazprom weekly/Gazprom Storage weekly {}.xlsx'.format(date.today()))

In [ ]:
minvalues = GazpromPlot['2015-01-05':'2019-12-29'].groupby(compare).min()
maxvalues = GazpromPlot['2015-01-05':'2019-12-29'].groupby(compare).max()
# avgvalues = GazpromPlot['2015-01-05':'2020'].groupby(compare).mean()
values2021 = GazpromPlot.loc['2021-01-05':'2022-01-02'].groupby(compare).mean()
values2022 = GazpromPlot.loc['2022-01-02':'2023-01-01'].groupby(compare).mean()
values2023 = GazpromPlot.loc['2022-01-02':].groupby(compare).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(minvalues,label='Minimum')
ax.plot(maxvalues,label='Maximum')
# ax.plot(avgvalues,label='Average')
ax.plot(values2021,label='2021')
ax.plot(values2022,label='2022')
ax.plot(values2023,label='2023')

plt.title('Total Gas Storage 2021 in Etzel, Jemgum, Katharina, Rehden, Haidach, Damborice compared with min/max 2015-2020',fontsize='16')
plt.ylabel('TWh',fontsize=12,rotation=0,labelpad=25)
plt.xlabel('Week Number',fontsize=12,rotation=0)
plt.legend()
plt.show()
fig.savefig('figures/Gazprom storage (monthly).png',dpi=300,bbox_inches="tight")

# Quality Assurance

In [ ]:
# Set the most recent date for which we have observations
last_date = str(today- timedelta(2))

In [ ]:
#url = 'https://agsi.gie.eu/api/data/21X000000001160J/AT'
# url = Points_APIs.iloc[0,5]
# r = requests.get(url)
# data = r.json()

In [ ]:
# These are normally the two APIs creating problems

# UGS Dambořice
# url = 'https://agsi.gie.eu/api/data/21W000000000102F/CZ/27X-MORAVIAGS--E'
# r = requests.get(url)
# print(r.status_code)    #500 means that the server encountered an unexpected condition that prevented it from fulfilling the request. This error is usually returned by the server when no other error code is suitable.

# # UGS Oude Statenzijl ETZEL Crystal
# url = 'https://agsi.gie.eu/api/data/21W000000000013E/DE/21X000000001125L'
# r = requests.get(url)
# print(r.status_code)     # The HTTP Status 200 (OK) status code indicates that the request has been processed successfully on the server. 

# UGS Etzel ESE (Gas Union Storage)
#Not existing

In [ ]:
# Gazprom Germany 
Etzel = ['UGS Oude Statenzijl ETZEL Crystal', 'UGS Etzel EGL (Equinor Storage Deutschland)',
         'UGS Etzel ESE (Gas Union Storage)', 'UGS Etzel ESE (OMV)',
         'UGS Etzel Erdgas Lager EGL', 'UGS Etzel ESE (Uniper Energy Storage)', 'UGS Etzel ESE (VGS)']
Jemgum   = ['UGS Jemgum H (astora)', 'VSP NORD (Rehden, Jemgum)', 'UGS Jemgum H (EWE)', 'UGS Jemgum H (VGS)']
Katharina = 'UGS Katharina'
Rehden = 'UGS Rehden'
# Gazprom Austria
Haidach = ['UGS Haidach (astora)', 'UGS Haidach (GSA)']
# Gazprom CZ
Damborice ='UGS Dambořice'

In [ ]:
df

In [ ]:
# df = df.set_index(pd.DatetimeIndex(df['dates']))
df_QA = df

In [ ]:
df_QA

In [ ]:
converter=0.0103
converter_GWh = converter* 1000
df_QA['storage in M3m'] = df_QA['values']/converter
df_QA['injection in M3m'] = df_QA['injection']/converter_GWh
df_QA['withdrawal in M3m'] = df_QA['withdrawal']/converter_GWh
df_QA['Max Storage in M3m'] = df_QA['Max Storage']/converter

In [ ]:
key_param = ["storage in M3m","injection in M3m","withdrawal in M3m","Max Storage in M3m"]

###  Etzel

In [ ]:
Etzel = ['UGS Oude Statenzijl ETZEL Crystal', 'UGS Etzel EGL (Equinor Storage Deutschland)',
         'UGS Etzel ESE (Gas Union Storage)', 'UGS Etzel ESE (OMV)',
         'UGS Etzel Erdgas Lager EGL', 'UGS Etzel ESE (Uniper Energy Storage)', 'UGS Etzel ESE (VGS)']

In [ ]:
df_Etzel = df_QA[df_QA.name.isin(Etzel)==True]
# del df_Etzel['dates']
df_Etzel = df_Etzel.groupby('dates')[key_param].sum().reset_index()
df_Etzel = df_Etzel.set_index(pd.DatetimeIndex(df_Etzel['dates']))

In [ ]:
df_Etzel

In [ ]:
df_Etzel[df_Etzel['dates']=='2019-12-01']

In [ ]:
df_QA[(df_QA['name']=='UGS Etzel EGL (Equinor Storage Deutschland)') & (df_QA.index =='2022-01-16')]

In [ ]:
df_QA[(df_QA['name']=='UGS Etzel ESE (OMV)') & (df_QA.index=='2022-01-16')]

In [ ]:
df_QA[(df_QA['name']=='UGS Etzel Erdgas Lager EGL') & (df_QA.index=='2022-01-16')]

In [ ]:
df_QA[(df_QA['name']=='UGS Etzel ESE (Uniper Energy Storage)') & (df_QA.index=='2022-01-16')]

In [ ]:
df_QA[(df_QA['name']=='UGS Etzel ESE (VGS)') & (df_QA.index=='2022-01-16')]

In [ ]:
for points in Etzel:
    cdtemp = df[df['name'] == points]['2022-01-01':'2022-07-18']
#     cdplot = cdtemp.groupby(pd.Grouper(freq='D'))#['2021-11-10':]

#     Ddata['{}'.format(country)] = cdplot #add data to Dataframe for export. 

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(cdtemp['values']/converter)

    plt.xticks(rotation=45)
    plt.title('storage in {}'.format(points),fontsize='16')
    plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

    plt.show()
#     fig.savefig('figures/DAILY_{}_imports.png'.format(country),dpi=300,bbox_inches="tight")

In [ ]:
df_QA

In [ ]:
Etzel = ['UGS Oude Statenzijl ETZEL Crystal', 'UGS Etzel EGL (Equinor Storage Deutschland)',
         'UGS Etzel ESE (Gas Union Storage)', 'UGS Etzel ESE (OMV)',
         'UGS Etzel Erdgas Lager EGL', 'UGS Etzel ESE (Uniper Energy Storage)', 'UGS Etzel ESE (VGS)']

startdate = '2019-06-01'
enddate = '2019-07-18'

## List of considered plants 
Crystal = df_QA[df_QA['name'] == 'UGS Oude Statenzijl ETZEL Crystal']
Crystalplot = Crystal['storage in M3m'][startdate:enddate]

EGL_Group = df_QA[df_QA['name'] == 'UGS Etzel EGL (Equinor Storage Deutschland)']
EGLplot = EGL_Group['storage in M3m'][startdate:enddate]

ESE_GUS = df_QA[df_QA['name'] == 'UGS Etzel ESE (Gas Union Storage)']
ESE_GUSplot = ESE_GUS['storage in M3m'][startdate:enddate]

ESE_OMV = df_QA[df_QA['name'] == 'UGS Etzel ESE (OMV)']
ESE_OMVplot = ESE_OMV['storage in M3m'][startdate:enddate]

EGL_Lager = df_QA[df_QA['name'] == 'UGS Etzel Erdgas Lager EGL']
EGL_Lagerplot = EGL_Lager['storage in M3m'][startdate:enddate]

ESE_Uniper = df_QA[df_QA['name'] == 'UGS Etzel ESE (Uniper Energy Storage)']
ESE_Uniperplot = ESE_Uniper['storage in M3m'][startdate:enddate]

VGS = df_QA[df_QA['name'] == 'UGS Etzel ESE (VGS)']
VGSplot = VGS['storage in M3m'][startdate:enddate]

## Code for figure plot
fig, ax = plt.subplots(figsize=(8, 8))

# ax.plot(Crystalplot,label='Crystal',linestyle=':',marker='o')
ax.plot(EGLplot,label='EGL group')
# ax.plot(ESE_GUSplot,label='ESE GUS',linestyle=':', marker='o')
ax.plot(ESE_OMVplot,label='ESE OMV')
ax.plot(EGL_Lagerplot,label='EGL Lager')
ax.plot(ESE_Uniperplot,label='ESE Uniper', color='b')
ax.plot(VGSplot,label='VGS',color='black')

# ax.plot(Crystalplot+EGLplot+ESE_GUSplot+ESE_OMVplot+EGL_Lagerplot+ESE_Uniperplot+VGSplot,label='Total')

date_form1 = DateFormatter("%m-%d")
ax.xaxis.set_major_formatter(date_form1)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.xticks(rotation=45)
plt.title('Etzel aggregation',fontsize='16')
plt.ylabel('Million \n cubic \n metres \n storage',fontsize=12,rotation=0)
plt.legend(loc='best')

plt.show()

In [ ]:
Etzel = ['UGS Oude Statenzijl ETZEL Crystal', 'UGS Etzel EGL (Equinor Storage Deutschland)',
         'UGS Etzel ESE (Gas Union Storage)', 'UGS Etzel ESE (OMV)',
         'UGS Etzel Erdgas Lager EGL', 'UGS Etzel ESE (Uniper Energy Storage)', 'UGS Etzel ESE (VGS)']

startdate = '2022-03-01'
enddate = '2022-07-17'

## List of considered plants 
Crystal = df_QA[df_QA['name'] == 'UGS Oude Statenzijl ETZEL Crystal']
Crystalplot = Crystal['storage in M3m'][startdate:enddate]

EGL_Group = df_QA[df_QA['name'] == 'UGS Etzel EGL (Equinor Storage Deutschland)']
EGLplot = EGL_Group['storage in M3m'][startdate:enddate]

ESE_GUS = df_QA[df_QA['name'] == 'UGS Etzel ESE (Gas Union Storage)']
ESE_GUSplot = ESE_GUS['storage in M3m'][startdate:enddate]

ESE_OMV = df_QA[df_QA['name'] == 'UGS Etzel ESE (OMV)']
ESE_OMVplot = ESE_OMV['storage in M3m'][startdate:enddate]

EGL_Lager = df_QA[df_QA['name'] == 'UGS Etzel Erdgas Lager EGL']
EGL_Lagerplot = EGL_Lager['storage in M3m'][startdate:enddate]

ESE_Uniper = df_QA[df_QA['name'] == 'UGS Etzel ESE (Uniper Energy Storage)']
ESE_Uniperplot = ESE_Uniper['storage in M3m'][startdate:enddate]

VGS = df_QA[df_QA['name'] == 'UGS Etzel ESE (VGS)']
VGSplot = VGS['storage in M3m'][startdate:enddate]

## Code for figure plot
fig, ax = plt.subplots(figsize=(8, 8))

# ax.plot(Crystalplot,label='Crystal',linestyle=':',marker='o')
ax.plot(EGLplot,label='EGL group')
# ax.plot(ESE_GUSplot,label='ESE GUS',linestyle=':', marker='o')
ax.plot(ESE_OMVplot,label='ESE OMV')
ax.plot(EGL_Lagerplot,label='EGL Lager')
ax.plot(ESE_Uniperplot,label='ESE Uniper',color='b')
ax.plot(VGSplot,label='VGS',linestyle=':')

# ax.plot(Crystalplot+EGLplot+ESE_GUSplot+ESE_OMVplot+EGL_Lagerplot+ESE_Uniperplot+VGSplot,label='Total',linestyle=':',marker='o')

date_form1 = DateFormatter("%m-%d")
# ax.xaxis.set_major_formatter(date_form1)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.xticks(rotation=45)
plt.title('Etzel aggregation',fontsize='16')
plt.ylabel('Million \n cubic \n metres \n storage',fontsize=12,rotation=0)
plt.legend(loc='best')

plt.show()

In [ ]:
# Set the most recent date for which we have observations
last_date = str(today- timedelta(2))

### Jemug

In [ ]:
Jemgum_ext   = ['UGS Jemgum H (astora)', 'VSP NORD (Rehden, Jemgum)', 'UGS Jemgum H (EWE)', 'UGS Jemgum H (VGS)']
Jemgum   = ['UGS Jemgum H (astora)', 'UGS Jemgum H (VGS)']

In [ ]:
# 'UGS Jemgum H (astora)'  + 'UGS Jemgum H (VGS)' 
Jemug_sum = 166+684
Jemug_sum

In [ ]:
df_Jemgum = df_QA[df_QA.name.isin(Jemgum)==True].copy()

In [ ]:
df_Jemgum_fill = df_Jemgum[df_Jemgum.index==last_date]
from numpy import average
weighted_avg = round(average(df_Jemgum_fill['fillness'], weights = df_Jemgum_fill['Max Storage']),2)
weighted_avg # same as the FT article

In [ ]:
df_Jemgum_fill

In [ ]:
# del df_Jemgum['dates']
# df_Jemgum = df_Jemgum.groupby('dates')[key_param].sum().reset_index()
# df_Jemgum = df_Jemgum.set_index(pd.DatetimeIndex(df_Jemgum['dates']))

In [ ]:
df_Jemgum[df_Jemgum.index==last_date]

In [ ]:
df_QA[(df_QA['name']=='UGS Jemgum H (EWE)') & (df_QA.index==last_date)]

In [ ]:
for points in Jemgum:
    cdtemp = df[df['name'] == points]['2022-03-01':last_date]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(cdtemp['values']/converter)

    plt.xticks(rotation=45)
    plt.title('storage in {}'.format(points),fontsize='16')
    plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

    plt.show()
#     fig.savefig('figures/DAILY_{}_imports.png'.format(country),dpi=300,bbox_inches="tight")

### Haidach

In [ ]:
df_Haidach = df_QA[df_QA.name.isin(Haidach)==True].copy()

In [ ]:
# del df_Haidach['dates']
# df_Haidach = df_Haidach.groupby('dates')[key_param].sum().reset_index()
# df_Haidach = df_Haidach.set_index(pd.DatetimeIndex(df_Haidach['dates']))

In [ ]:
df_Haidach[df_Haidach.index==last_date]

In [ ]:
df_Haidach_fill = df_Haidach[df_Haidach.index==last_date]
from numpy import average
weighted_avg = round(average(df_Haidach_fill['fillness'], weights = df_Haidach_fill['Max Storage']),2)
weighted_avg # same as the FT article

In [ ]:
for points in Haidach:
    cdtemp = df[df['name'] == points]['2022-01-01':]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(cdtemp['values']/converter)

    plt.xticks(rotation=45)
    plt.title('storage in {}'.format(points),fontsize='16')
    plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

    plt.show()
#     fig.savefig('figures/DAILY_{}_imports.png'.format(country),dpi=300,bbox_inches="tight")

### after Gazprom expropriation some reporting changed hat

#### Austria

In [ ]:
new_Haidach = df[df['name']== 'RAG Storage Pool (Puchkirchen / Haag, Aigelsbrunn, Haidach 5, 7Fields-RAG)']

In [ ]:
cdtemp = new_Haidach['2020-01-01':]

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(cdtemp['values']/converter)

plt.xticks(rotation=45)
plt.title('storage in RAG Storage Pool'.format(points),fontsize='16')
plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

plt.show()
#     fig.savefig(

### Katharina

In [ ]:
cdtemp = df[df['name'] == 'UGS Katharina']['2019-01-01':]

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(cdtemp['values']/converter)

plt.xticks(rotation=45)
plt.title('storage in Katharina',fontsize='16')
plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

plt.show()
#     fig.savefig('figures/DAILY_{}_imports.png'.format(country),dpi=300,bbox_inches="tight")

In [ ]:
df[df['name'] == 'UGS Katharina']['2022-01-01':]

### Rehden

In [ ]:
last_date = str(today- timedelta(2))

In [ ]:
df_QA[(df_QA['name']=='UGS Rehden') & (df_QA.index==last_date)]
# same as FT article

In [ ]:
cdtemp = df[df['name'] == 'UGS Rehden']['2022-01-01':]

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(cdtemp['values']/converter)

plt.xticks(rotation=45)
plt.title('storage in Rehden',fontsize='16')
plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

plt.show()
#     fig.savefig('figures/DAILY_{}_imports.png'.format(country),dpi=300,bbox_inches="tight")

### Dambořice

In [ ]:
#Dambořice
cdtemp = df[df['name'] == 'UGS Dambořice'][:]

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(cdtemp['values']/converter)

plt.xticks(rotation=45)
plt.title('storage in Dambořice',fontsize='16')
plt.ylabel('Million \n cubic \n metres',fontsize=12,rotation=0)

plt.show()

# Rubbish?

In [ ]:
Gazprom_aggregate = [Etzel,Jemgum,Katharina,Rehden,Haidach,Damborice]

In [ ]:
Gazprom_aggregate = pd.DataFrame(Gazprom_aggregate)
Gazprom_aggregate.to_excel(r'C:\Users\giovanni.sgaravatti\OneDrive - Bruegel\Documents\Gas flow model\AGSI storage\Gazprom aggregate.xlsx')

In [ ]:
Gazprom_APIs = pd.read_excel(r'C:\Users\giovanni.sgaravatti\OneDrive - Bruegel\Documents\Gas flow model\AGSI storage\Gazprom APIs.xlsx')

In [ ]:
dates = []
values = []
injection = []
withdrawal =[]
percentage = []
status = []
max_storage = []
country = []
Name = []

for item in range(len(Gazprom_APIs)):
        url = Gazprom_APIs.iloc[item,5]
        try:
            r = requests.get(url)
            if r.status_code != 200:
                print(r.status_code)
                print(url)

            data = r.json()
            for x in data:
                #here we work with the APIs
                date = x['gasDayStartedOn']
                value = x['gasInStorage']
                inj = x['injection']
                withdr = x['withdrawal']
                percentages = x['full']
                statuses = x['status']
                Max_Storage = x['workingGasVolume']
                #here we work with the excel file
                countries = Points_APIs.iloc[item,0]
                Names = Points_APIs.iloc[item,4]


                dates.append(date)     
                values.append(value)
                injection.append(inj)
                withdrawal.append(withdr)
                percentage.append(percentages)
                status.append(statuses)
                max_storage.append(Max_Storage)
                country.append(countries)
                Name.append(Names)
        except Exception as e:
            print(e)

In [ ]:
df_Gazprom_check = pd.DataFrame()
df_Gazprom_check['dates'] = dates
df_Gazprom_check['values'] = values
df_Gazprom_check['injection'] = injection
df_Gazprom_check['withdrawal'] = withdrawal
df_Gazprom_check['fillness'] = percentage
df_Gazprom_check['status'] = status
df_Gazprom_check['Max Storage'] = max_storage
df_Gazprom_check['country'] = country
df_Gazprom_check['name'] = Name

In [ ]:
# This doesn't work Unable to parse string "-" at position 11862
# Getting rid of observations for which we have no data
df_Gazprom_check = df[(df['status']!='N') & (df['status']!='E')]
# Getting rid of NaNa
df_Gazprom_check.replace('', np.NaN)  #we set missing values to the numpy notation, NaN - easier to work with. 
df_Gazprom_check.replace('-', np.NaN)
# #We need to set it to integer to work with. 

In [ ]:
Gazprom_countries =['AT','CZ', 'DE'] 

In [ ]:
# EU_Gazprom = df[df['country'].isin(Gazprom_countries)]
# Gazprom_list = list(EU_Gazprom['name'].unique())
# Gazprom_list

In [ ]:
# Gazprom_list = pd.DataFrame(Gazprom_list)
# Gazprom_list.to_excel(r'C:\Users\giovanni.sgaravatti\OneDrive - Bruegel\Documents\Gas flow model\AGSI storage\Gazprom list.xlsx')

## This works but calendar day (day 1 is the first of the month)

In [ ]:
values_30_days['daily'] = range(1,32)  # I tried to be smart with groupby() by I didn't manage to
values_30_days = values_30_days.set_index(values_30_days['daily']) 
del values_30_days['daily']

In [ ]:
Gas_value = df.groupby(['country','dates'])['values'].sum().reset_index()

In [ ]:
Gas_value = Gas_value.set_index(pd.DatetimeIndex(Gas_value['dates']))

In [ ]:
Gas_value[Gas_value['country']=='PL']

In [ ]:
country = 'PL'
startdate = '2015-01'
enddate = '2020-12'

tick_spacing = 400
date_form = DateFormatter("%Y")

dfcountry = Gas_value[Gas_value['country'] == country]
                              
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(dfcountry.groupby(pd.Grouper(freq='M')).values,label='value')

ax.xaxis.set_major_formatter(date_form)
ax.xaxis.set_major_locator(ticker.MultipleLocator(tick_spacing))
ax.yaxis.set_label_coords(-0.15,.45)

plt.title('Country: {} and measure: {}'.format(country,'value'),fontsize=20)
plt.ylabel('TWh',fontsize=12,rotation=0) 
                                                                   
plt.legend()
# fig.savefig('plantsM/{}.png'.format('Beregdaróc'),dpi=300,bbox_inches="tight")
# plt.show()
